# 분류: 콘크리트 균열

이미지 한 장을 보고 균열이 있는지(`crack`) 없는지만(`intact`) 구분한다. 모델이 어디를 봤는지는 Grad-CAM으로 확인한다.

**런타임 → 런타임 유형 변경 → GPU(T4)** 를 먼저 고른다.


## 1. 데이터 zip

드라이브에서 해당 zip을 `링크가 있는 모든 사용자`로 공유한 뒤, 링크를 `ZIP_URL`에 넣는다.

파일이 100MB보다 크면 `wget`이 구글 확인 페이지를 받는다. 드라이브 링크는 `gdown`, 직접 열리는 주소는 `wget`을 쓴다.


In [ ]:
ZIP_URL = "https://drive.google.com/file/d/1c5ca46KgfBWQF2IvQvlS5PTRCMWzYcw5/view?usp=sharing"

TASK = "classification"
ZIP_PATH = "/content/01_classification.zip"
DST = f"/content/{TASK}"


## 2. 패키지


In [ ]:
!pip install -q gdown


## 3. zip 받기


In [ ]:
from pathlib import Path
import os


def fetch_zip(url, dest):
    url = (url or "").strip()
    if not url or "여기에" in url:
        raise SystemExit(
            "위 셀 ZIP_URL에 구글 드라이브 공유 링크(또는 직접 zip 주소)를 넣는다."
        )

    is_drive = "drive.google.com" in url or "docs.google.com" in url

    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 10_000:
        print("이미 받은 파일이 있다:", dest, dest.stat().st_size, "bytes")
        return

    dest.parent.mkdir(parents=True, exist_ok=True)

    os.environ["ZIP_URL"] = url
    os.environ["ZIP_PATH"] = str(dest)
    if is_drive:
        !gdown --fuzzy "$ZIP_URL" -O "$ZIP_PATH"
    else:
        !wget -c -O "$ZIP_PATH" "$ZIP_URL"

    print("받음", dest, dest.stat().st_size, "bytes")


fetch_zip(ZIP_URL, ZIP_PATH)


In [ ]:
!rm -rf {DST}


In [ ]:
!unzip -q -o {ZIP_PATH} -d /content


In [ ]:
from pathlib import Path
import torch
from IPython.display import display
from PIL import Image

ROOT = Path(DST)
if not ROOT.exists():
    raise SystemExit(f"{ROOT} 가 없다. zip 안에 과제 폴더가 있는지 확인한다.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("ROOT", ROOT)
print("torch", torch.__version__)
print("device", DEVICE)
if DEVICE.type == "cuda":
    print("gpu", torch.cuda.get_device_name(0))


def show_img(img, width=900):
    """셀 결과에 PIL 이미지를 띄운다. 너무 가로로 긴 장만 줄인다."""
    img = img.convert("RGB")
    w, h = img.size
    if w > width:
        img = img.resize((width, int(h * width / w)), Image.BILINEAR)
    display(img)


## 4. 모델

ImageNet으로 미리 학습한 torchvision ResNet18의 마지막 층만 2클래스로 바꾼다. 이 장은 공식 구현을 가져온다.


In [ ]:
from torch import nn
from torchvision.models import ResNet18_Weights, resnet18


def build_resnet18(num_classes=2, pretrained=True):
    weights = ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
    model = resnet18(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model

## 5. 학습

검증 정확도로 `best.pt`를 고른다. 공식 test 8,000장은 학습과 모델 선택에 쓰지 않고, 학습이 끝난 뒤에 한 번만 평가한다.


In [ ]:
import json
import random

import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

CLASSES = ("crack", "intact")
EPOCHS = 10
BATCH = 32
IMGSZ = 224
LR = 1e-3
SEED = 42


def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def accuracy(logits, y):
    return (logits.argmax(1) == y).float().mean().item()


def run_epoch(model, loader, criterion, optimizer, train):
    model.train(train)
    total_loss, total_acc, n = 0.0, 0.0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.set_grad_enabled(train):
            logits = model(x)
            loss = criterion(logits, y)
            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

        bs = y.size(0)
        total_loss += loss.item() * bs
        total_acc += accuracy(logits.detach(), y) * bs
        n += bs

    return total_loss / n, total_acc / n


def eval_split(model, loader, classes):
    model.eval()
    correct = {c: 0 for c in classes}
    total = {c: 0 for c in classes}
    n_ok, n = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            pred = model(x).argmax(1)
            n += y.size(0)
            n_ok += int((pred == y).sum().item())
            for t, p in zip(y.tolist(), pred.tolist()):
                name = classes[t]
                total[name] += 1
                if t == p:
                    correct[name] += 1

    per = {c: (correct[c] / total[c] if total[c] else 0.0) for c in classes}
    return {"acc": n_ok / n if n else 0.0, "n": n, "per_class": per, "counts": total}


mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]
train_tf = transforms.Compose(
    [
        transforms.Resize((IMGSZ, IMGSZ)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(0.15, 0.15),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ]
)
eval_tf = transforms.Compose(
    [
        transforms.Resize((IMGSZ, IMGSZ)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ]
)

set_seed(SEED)
data = ROOT / "dataset"
runs = ROOT / "runs"
train_ds = datasets.ImageFolder(data / "train", transform=train_tf)
val_ds = datasets.ImageFolder(data / "val", transform=eval_tf)
if tuple(train_ds.classes) != CLASSES:
    raise SystemExit(f"클래스 순서가 {CLASSES}여야 하는데 {train_ds.classes}이다.")

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=0)

model = build_resnet18(num_classes=len(CLASSES)).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
runs.mkdir(parents=True, exist_ok=True)

best_acc = -1.0
for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer, True)
    va_loss, va_acc = run_epoch(model, val_loader, criterion, optimizer, False)
    print(
        f"epoch {epoch:03d}  train loss {tr_loss:.4f} acc {tr_acc:.3f}  "
        f"val loss {va_loss:.4f} acc {va_acc:.3f}"
    )
    if va_acc >= best_acc:
        best_acc = va_acc
        torch.save(
            {"model": model.state_dict(), "classes": list(CLASSES), "imgsz": IMGSZ},
            runs / "best.pt",
        )

print("saved", runs / "best.pt", "best_val_acc", round(best_acc, 4))

ckpt = torch.load(runs / "best.pt", map_location="cpu", weights_only=True)
model.load_state_dict(ckpt["model"])
test_ds = datasets.ImageFolder(data / "test", transform=eval_tf)
test_loader = DataLoader(test_ds, batch_size=BATCH, shuffle=False, num_workers=0)
print("held_out_test", eval_split(model, test_loader, CLASSES))


## 6. 추론과 Grad-CAM

왼쪽이 입력, 가운데가 모델이 본 위치, 오른쪽이 클래스 확률이다. 그림은 셀 아래에 바로 뜬다.


In [ ]:
import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFont
from torchvision import transforms


def font(size):
    for path in (
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf",
    ):
        try:
            return ImageFont.truetype(path, size)
        except OSError:
            continue
    return ImageFont.load_default()


def gradcam(model, x, class_idx):
    acts = []
    grads = []

    def fwd(_m, _i, o):
        acts.append(o)

    def bwd(_m, _gi, go):
        grads.append(go[0])

    h1 = model.layer4.register_forward_hook(fwd)
    h2 = model.layer4.register_full_backward_hook(bwd)
    model.zero_grad(set_to_none=True)
    logits = model(x)
    logits[0, class_idx].backward()
    h1.remove()
    h2.remove()

    w = grads[0][0].mean(dim=(1, 2))
    cam = (w[:, None, None] * acts[0][0]).sum(0).relu()
    cam = cam / (cam.max() + 1e-8)
    return cam.detach().cpu().numpy()


def jet_rgb(cam):
    x = np.clip(cam, 0.0, 1.0)
    r = np.clip(1.5 - np.abs(4 * x - 3), 0, 1)
    g = np.clip(1.5 - np.abs(4 * x - 2), 0, 1)
    b = np.clip(1.5 - np.abs(4 * x - 1), 0, 1)
    rgb = np.stack([r, g, b], axis=-1)
    return Image.fromarray((rgb * 255).astype(np.uint8), mode="RGB")


def overlay_cam(img, cam):
    heat = Image.fromarray((cam * 255).astype(np.uint8), mode="L").resize(img.size, Image.BILINEAR)
    color = jet_rgb(np.array(heat, dtype=np.float32) / 255.0).resize(img.size, Image.BILINEAR)
    return Image.composite(Image.blend(img, color, 0.55), img, heat)


def draw_probs(classes, prob, pred, width, height):
    panel = Image.new("RGB", (width, height), (18, 18, 18))
    draw = ImageDraw.Draw(panel)
    draw.text((16, 16), f"pred: {classes[pred]}", fill=(255, 255, 255), font=font(18))

    top = 56
    bar_w = width - 40
    row_h = max(48, (height - top - 20) // max(len(classes), 1))
    for i, name in enumerate(classes):
        y = top + i * row_h
        p = float(prob[i])
        fill = (40, 180, 90) if i == pred else (70, 70, 70)
        draw.rectangle([16, y, 16 + int(bar_w * p), y + 22], fill=fill)
        draw.rectangle([16, y, 16 + bar_w, y + 22], outline=(120, 120, 120))
        draw.text((16, y + 26), f"{name}  {p:.3f}", fill=(230, 230, 230), font=font(16))

    return panel


def visualize(raw, cam_ov, classes, prob, pred):
    h, w = 420, 420
    left = raw.resize((w, h), Image.BILINEAR)
    mid = cam_ov.resize((w, h), Image.BILINEAR)
    right = draw_probs(list(classes), prob, pred, 280, h)
    canvas = Image.new("RGB", (w * 2 + 280, h))
    canvas.paste(left, (0, 0))
    canvas.paste(mid, (w, 0))
    canvas.paste(right, (w * 2, 0))
    return canvas


weights = ROOT / "runs" / "best.pt"
ckpt = torch.load(weights, map_location="cpu", weights_only=True)
classes = ckpt["classes"]
imgsz = ckpt.get("imgsz", 224)

model = build_resnet18(num_classes=len(classes), pretrained=False)
model.load_state_dict(ckpt["model"])
model.eval()

tf = transforms.Compose(
    [
        transforms.Resize((imgsz, imgsz)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)

images = sorted(
    p
    for p in (ROOT / "examples").glob("*.jpg")
    if not p.name.endswith(("_pred.jpg", "_cam.jpg", "_gt.jpg"))
)

for path in images:
    raw = Image.open(path).convert("RGB")
    x = tf(raw).unsqueeze(0)
    x.requires_grad_(True)
    logits = model(x)
    prob = logits.softmax(1)[0].detach()
    pred = int(prob.argmax())
    cam = gradcam(model, x, pred)
    canvas = visualize(raw, overlay_cam(raw, cam), classes, prob, pred)

    print(path.name, "pred=", classes[pred])
    for i, name in enumerate(classes):
        mark = " <--" if i == pred else ""
        print(f"  {name:8s} {prob[i].item():.3f}{mark}")

    gt = path.parent / f"{path.stem}_gt.jpg"
    if gt.exists():
        print("정답")
        show_img(Image.open(gt), width=720)

    print("예측")
    show_img(canvas, width=980)
    print()
